# Python как рабочий инструмент AI/ML Engineer

В этом ноутбуке рассматриваются:
- простые структуры данных Python в контексте проекта;
- работа с путями и файлами через `pathlib`;
- чтение табличных данных в `pandas`;
- объединение таблиц и построение признаков;
- переход от табличной логики к численным вычислениям через `numpy`;
- комплексный прикладной пример на данных проекта.


## 1. Простые структуры данных Python

Начнём с базовых структур языка: строк, списков и словарей.

На этом примере важно увидеть, что даже простые объекты Python в AI/ML-проекте имеют прикладной смысл: строки могут хранить имена и категории, списки — признаки, а словари — описание объекта.


In [1]:
customer_name = "Анна"
category = "Электроника"
file_path = "data/orders.csv"

features = ["amount", "category", "city", "segment"]

customer_info = {
    "customer_id": 1,
    "name": "Анна",
    "city": "Москва",
    "segment": "B2C"
}

print("Строка:", customer_name)
print("Категория:", category)
print("Путь к файлу:", file_path)
print("Список признаков:", features)
print("Словарь с данными клиента:", customer_info)


Строка: Анна
Категория: Электроника
Путь к файлу: data/orders.csv
Список признаков: ['amount', 'category', 'city', 'segment']
Словарь с данными клиента: {'customer_id': 1, 'name': 'Анна', 'city': 'Москва', 'segment': 'B2C'}


## 2. Подключение библиотек

Подключим библиотеки, которые понадобятся для работы с путями, таблицами и численными вычислениями.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np


## 3. Работа с путями и файлами

Теперь покажем, как Python работает с файловой структурой проекта.

Для этого используем модуль `pathlib`, который позволяет аккуратно формировать пути к данным и проверять существование файлов.


In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"

orders_path = DATA_DIR / "orders.csv"
customers_path = DATA_DIR / "customers.csv"

print("BASE_DIR =", BASE_DIR)
print("orders_path =", orders_path)
print("customers_path =", customers_path)
print("orders.csv существует:", orders_path.exists())
print("customers.csv существует:", customers_path.exists())


## 4. Загрузка данных в Pandas

На этом этапе CSV-файлы превращаются в объекты `DataFrame`.

Именно с такими табличными структурами обычно начинается практическая работа с данными в Python.


In [ ]:
orders_df = pd.read_csv(orders_path)
customers_df = pd.read_csv(customers_path)

print("Тип orders_df:", type(orders_df))
orders_df.head()


## 5. Несколько источников данных

В реальных проектах данные обычно хранятся не в одной таблице, а в нескольких источниках.

Поэтому отдельно посмотрим таблицу клиентов, которую затем объединим с таблицей заказов.


In [ ]:
customers_df.head()


## 6. Объединение таблиц

Теперь объединим данные о заказах и клиентах.

Это типичная операция подготовки данных: чтобы получить более полное представление об объектах, данные из разных таблиц нужно соединить по общему ключу.


In [ ]:
df = orders_df.merge(customers_df, on="customer_id", how="left")
df.head()


## 7. Построение признаков

На этом этапе переходим от сырых данных к более полезному представлению.

Для каждого клиента рассчитаем агрегированные характеристики: общую сумму покупок, количество заказов и средний чек.


In [ ]:
customer_features = (
    df[df["status"] == "paid"]
      .groupby("customer_id")
      .agg(
          total_spent=("amount", "sum"),
          orders_count=("order_id", "count"),
          avg_check=("amount", "mean")
      )
      .reset_index()
)

customer_features


## 8. Функция как этап обработки данных

Ту же самую логику теперь оформим в виде функции.

Это позволяет превратить отдельный фрагмент кода в самостоятельный и переиспользуемый шаг пайплайна обработки данных.


In [ ]:
def build_customer_features(orders_df, customers_df):
    paid_orders = orders_df[orders_df["status"] == "paid"].copy()
    merged = paid_orders.merge(customers_df, on="customer_id", how="left")
    return (
        merged.groupby("customer_id")
              .agg(
                  total_spent=("amount", "sum"),
                  orders_count=("order_id", "count"),
                  avg_check=("amount", "mean")
              )
              .reset_index()
    )


features_df = build_customer_features(orders_df, customers_df)
features_df


## 9. Переход от таблицы к численному массиву

Теперь выделим числовой столбец и преобразуем его в массив NumPy.

**Обратите внимание:** табличная структура удобна для анализа данных, а массив — для численных вычислений.


In [ ]:
amounts = df["amount"].to_numpy()

print("Тип amounts:", type(amounts))
print("Массив amounts:", amounts)
print("Среднее значение:", amounts.mean())
print("Максимум:", amounts.max())


## 10. Векторизация вычислений

Покажем, как NumPy позволяет выполнять операции сразу над всем массивом.

Это один из ключевых принципов эффективной работы с численными данными в Python.


In [ ]:
discounted_amounts = amounts * 0.9
discounted_amounts


## 11. Фильтрация по условию

Теперь посмотрим, как можно выбирать элементы массива по условию.

Для этого используется булева маска — важный инструмент фильтрации в NumPy.


In [ ]:
high_value_mask = amounts > 1500
amounts[high_value_mask]


## 12. Комплексный прикладной пример

Теперь объединим уже рассмотренные приёмы в единый сценарий.

Задача: из таблиц заказов и клиентов построить клиентскую витрину, добавить простой сегмент и сохранить результат в отдельный файл.


### 12.1. Построение клиентской витрины

Сначала используем ранее созданную функцию, чтобы получить агрегированные признаки по клиентам.

На этом шаге мы переходим от сырых заказов к более удобному аналитическому представлению данных.


In [ ]:
customer_features = build_customer_features(orders_df, customers_df)
customer_features


### 12.2. Простая сегментация клиентов

Теперь добавим интерпретируемый результат: если суммарные траты клиента превышают заданный порог, присваиваем ему сегмент `VIP`, иначе — `Regular`.

Это показывает, как числовые признаки можно преобразовать в прикладную бизнес-логику.


In [ ]:
customer_features["segment_label"] = np.where(
    customer_features["total_spent"] > 3000,
    "VIP",
    "Regular"
)

customer_features


### 12.3. Сортировка клиентов по ценности

Отсортируем клиентов по общей сумме трат.

Такой шаг помогает быстро увидеть наиболее ценных клиентов и уже ближе к реальной аналитической задаче.


In [ ]:
top_customers = customer_features.sort_values("total_spent", ascending=False)
top_customers


### 12.4. Отбор нужного сегмента

После сегментации можно быстро выделить только ту группу клиентов, которая нас интересует.

Здесь мы выбираем из общей витрины только клиентов сегмента `VIP`.


In [ ]:
vip_customers = customer_features[customer_features["segment_label"] == "VIP"]
vip_customers


### 12.5. Сохранение результата

Теперь сохраним итоговую витрину в отдельный CSV-файл.

Это важный практический шаг: результат обработки данных часто нужно не только посмотреть на экране, но и сохранить для дальнейшего использования.


In [ ]:
output_path = DATA_DIR / "customer_features_from_notebook.csv"
customer_features.to_csv(output_path, index=False)

print("Файл сохранён:", output_path)


### 12.6. Проверка результата

Сразу проверим, что сохранённый файл можно корректно считать обратно.

Это завершает небольшой прикладной цикл: от исходных данных — к построению признаков, сегментации и сохранению результата.


In [ ]:
saved_features_df = pd.read_csv(output_path)
saved_features_df


## 13. Вывод по комплексному примеру

Python позволяет объединить в одном сценарии:
- чтение исходных данных;
- объединение таблиц;
- построение признаков;
- применение правил сегментации;
- сохранение результата.

Именно поэтому Python удобен как рабочая среда для прикладной аналитики и подготовки данных в AI/ML-задачах.
